# 第 2 章练习：深度学习与神经网络
这份练习不是考察 API 记忆，而是帮助你独立建立三种能力：看懂梯度、完成训练循环、追踪 CNN 中的数据形状。

建议先关闭完整实验。每项任务都先阅读**为什么做**和**运行后观察什么**，再完成 TODO，最后运行统一验收。

## 准备运行环境

**为什么运行这段代码？** 后续任务同时需要数值计算、PyTorch 神经网络和 Wine 数据集。这里一次性导入工具，并固定随机种子，使模型初始化和实验结果尽量可复现。

**你要学什么？** `numpy` 负责数组处理，`torch` 负责张量、自动求导与神经网络，`train_test_split` 负责建立互不混用的训练集和测试集。这里不训练模型，只是在准备实验工具。

In [ ]:
from pathlib import Path
import numpy as np
import torch
from torch import nn
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
torch.manual_seed(42)

# 任务一：补全 Autograd

### 为什么做这个任务
训练神经网络时，优化器需要知道每个参数对 Loss 的影响方向。`backward()` 沿计算图应用链式法则，把导数写入张量的 `.grad`。理解这一点，训练循环才不是需要死记的四行代码。

### 你需要完成什么
1. 使用已经创建的 `x` 构造 $z=x^2+x^2/2$；
2. 对 `z` 调用 `backward()`；
3. 从 `x.grad` 取出普通数值，赋给 `gradient_value`。

运行前先手算：$dz/dx=2x+x$。运行后比较手算和 Autograd 是否一致。

<details><summary>需要提示时点这里</summary>先写 `z = ...`，再调用 `z.backward()`；标量张量可使用 `.item()` 取出 Python 数值。</details>

In [ ]:
x=torch.tensor(2.0,requires_grad=True)
# TODO：构造 z 并反向传播
z=None
gradient_value=None

# 任务二：训练 MLP

### 为什么做这个任务
上一任务只对一个标量求导。这里把同一机制扩展到真正的神经网络：Forward 产生 logits，Loss 衡量错误，Backward 计算所有参数的梯度，Optimizer 根据梯度更新参数。

前四行已经完成数据划分和标准化。均值、标准差只从训练集计算，是为了避免测试集信息泄漏。`xt/yt` 用于训练，`xv` 只用于最后评估。

### 你需要完成什么
1. 创建 `13 → 隐藏层 → 3` 的 MLP，并加入非线性激活；
2. 创建 `CrossEntropyLoss` 和优化器；
3. 重复执行 `zero_grad → forward → loss → backward → step`；
4. 切换到评估模式，在 `no_grad()` 下预测测试集；
5. 对 logits 使用 `argmax(dim=1)` 得到类别编号并转成 NumPy。

**运行后观察什么？** `mlp_predictions` 的 Shape 应与 `mlp_targets` 相同。输出层不要添加 Softmax，因为交叉熵直接接收 logits。

<details><summary>需要提示时点这里</summary>可从 `Linear(13, 24) → ReLU → Linear(24, 3)` 和 Adam 开始，训练约 200～300 次。每轮更新前必须清空旧梯度。</details>

In [ ]:
# 已提供部分：加载数据并建立固定的分层划分
features, targets = load_wine(return_X_y=True)
all_indices = np.arange(len(targets))
train_indices, test_indices = train_test_split(
    all_indices, test_size=0.25, random_state=42, stratify=targets
)

# 已提供部分：标准化统计量只能来自训练集
train_mean = features[train_indices].mean(axis=0)
train_std = features[train_indices].std(axis=0) + 1e-7
train_inputs = torch.tensor(
    (features[train_indices] - train_mean) / train_std, dtype=torch.float32
)
train_targets = torch.tensor(targets[train_indices], dtype=torch.long)
test_inputs = torch.tensor(
    (features[test_indices] - train_mean) / train_std, dtype=torch.float32
)

# TODO：创建并训练 MLP
mlp_model = None
mlp_predictions = None
mlp_targets = targets[test_indices]

# 任务三：追踪 CNN Shape

### 为什么做这个任务
CNN 最常见的问题不是语法，而是层间 Shape 对不上。图像张量采用 `[Batch, Channel, Height, Width]`；卷积通常改变 Channel，池化缩小空间尺寸，分类头最终输出 `[Batch, Classes]`。

### 你需要完成什么
1. 创建至少包含 `Conv2d`、激活函数和池化的 CNN；
2. 把 `cnn_input` 传过网络执行一次 Forward；
3. 确保最终输出 Shape 为 `[8, 10]`；
4. 使用 `detach().numpy()` 得到 `cnn_output`。

**运行后观察什么？** 8 是样本数，10 是类别数；输出是每个样本的 10 个 logits，不是最终类别。

<details><summary>需要提示时点这里</summary>`AdaptiveAvgPool2d(1) → Flatten() → Linear(...)` 可以避免手算分类头前的空间尺寸，但你仍应打印每层 Shape 并解释变化。</details>

In [ ]:
cnn_input=torch.randn(8,1,28,28)
# TODO：创建 CNN，执行 Forward，并赋值 cnn_output
cnn_model=None
cnn_output=None

# 统一验收

这段代码不负责训练，而是读取当前 Notebook 中的三个成果变量，分别验证梯度、MLP 预测质量和 CNN 输出 Shape。全部通过后才生成 `artifacts/exercise_result.json`。

如果失败，先阅读对应检查消息，不要直接填写期望输出；验收目标是证明前面的计算过程确实产生了正确结果。

In [ ]:
from validate import validate_exercise,print_report,save_report
exercise_report=validate_exercise(globals()); print_report(exercise_report)
if exercise_report['passed']: save_report(exercise_report,Path('artifacts/exercise_result.json'))